# Deep Learning for Geo/Environmental sciences

<center><img src="../logo_2.png" alt="logo" width="500"/></center>

<em>*Created with ChatGPT</em>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/climate-analytics-lab/sioc209-2026-sp/blob/main/sioc209-2026-sp/06_unsupervised_learning/13_contrastive_learning_example.ipynb)

## Lecture 13: Contrastive Learning

 - [Recap](#Recap)
 - [Contrastive Learning](#Contrastive-Learning)
 - [SimCLR](#SimCLR)
 - [Tile2Vec](#Tile2Vec)
 - [Applying Contrastive Learning to Remote Sensing](#Applying-Contrastive-Learning-to-Remote-Sensing)
 - [Beyond SimCLR](#Beyond-SimCLR)
 - [From SSL to Foundation Models](#From-SSL-to-Foundation-Models)

## Recap

In the last lecture we introduced unsupervised learning and discussed dimensionality reduction techniques. We learned about three feature extraction techniques: PCA, t-SNE, and Autoencoders.

These are distinct from feature selection techniques, which select a subset of the original features. Feature extraction techniques transform the original features into a new set of features.

As unsupervised learning techniques, they do not require labeled data. They are used to reduce the dimensionality of the data, which can help in visualizing the data, removing noise, and improving the performance of machine learning models.

## Recap

While PCA will preserve the global structure of the data, t-SNE will preserve the local structure of the data. This makes t-SNE particularly well-suited for visualizing clusters in high-dimensional data. 



Both techniques have their strengths and weaknesses, and the choice of which technique to use will depend on the specific characteristics of the data and the goals of the analysis.

Autoencoders are a type of neural network that are trained to learn a compressed representation of the input data. We showed how to build an autoencoder using Keras and how to use it for dimensionality reduction and denoising.

### Unsupervised, Semi-Supervised, and Self-Supervised Learning

As we discussed in the last lecture, unsupervised learning is a type of machine learning that involves training models on data without labels. This is in contrast to supervised learning, where models are trained on labeled data, and reinforcement learning, where models are trained through trial and error.



Unsupervised learning can be further divided into unsupervised, semi-supervised, and self-supervised learning:
 - In unsupervised learning, models are trained on data without labels. 
 - In semi-supervised learning, models are trained on a combination of labeled and unlabeled data. 
 - In self-supervised learning, models are trained on data that is automatically labeled by the model itself.



In this lecture, we will focus on self-supervised learning, and in particular, contrastive learning.



### Pre-text Tasks

In self-supervised learning, models are trained on pre-text tasks. These are tasks that are designed to provide supervision to the model without requiring human-labeled data. The model is trained to solve these pre-text tasks, and in the process, it learns useful representations of the data that can be transferred to downstream tasks.



Pre-text tasks can take many forms, such as predicting the rotation of an image, predicting the relative position of patches in an image, or predicting the color of a grayscale image. The key idea is that the pre-text task should be designed in such a way that the model learns useful representations of the data.



## Contrastive Learning

Contrastive learning is a technique for learning representations of data by contrasting similar and dissimilar pairs of data points. The idea is to learn a representation that brings similar data points closer together in the embedding space and pushes dissimilar data points further apart.



It is typically used in unsupervised learning and self-supervised learning, where the goal is to learn representations of data without the need for labeled data, and often as a pretraining (or pretext) step for supervised learning tasks.



The idea of pretraining is to learn a good representation of the data in an unsupervised manner, and then fine-tune this representation on a supervised task. This can help to improve the performance of the model on the supervised task, especially when labeled data is scarce. 


## Contrastive Learning

A key element of contrastive learning is choosing the augmentation strategy. The augmentation strategy is used to create positive and negative pairs of data points for the contrastive loss. Positive pairs are pairs of data points that are similar, while negative pairs are pairs of data points that are dissimilar.

In choosing the augmentation strategy we are providing the model with a way to learn the invariances in the data. For example, in the case of images, we might use random crops, rotations, flips, and color distortions as augmentations. These augmentations help the model to learn to recognize objects regardless of their position, orientation, or color.

In this way we are implicitly teaching the model about the underlying structure of the data, without needing to provide explicit labels. This is the key idea behind contrastive learning. Let's look at two different approaches for this.

### SimCLR

A Simple Framework for Contrastive Learning of Visual Representations (SimCLR) is a contrastive learning technique that learns visual representations of images by contrasting similar and dissimilar pairs of images. The technique is based on the idea of learning a representation of the image that captures the underlying structure of the image.



It was introduced by Chen et al. in 2020 and has nearly 17000 citations as of today: https://arxiv.org/abs/2002.05709



SimCLR works by training a neural network to predict the similarity of pairs of images. The network is trained using a contrastive loss function, which encourages the network to learn representations that are close together for similar images and far apart for dissimilar images.



The result is a set of visual representations of the images that capture the underlying structure of the images. These representations can be used for a variety of tasks, such as image retrieval, image classification, and image segmentation.



<center><img src="_images/simCLR.gif" alt="SimCLR" width="600"/></center>


### SimCLR
The key components of SimCLR are:

1. Data augmentation: SimCLR uses a variety of data augmentation techniques to generate pairs of similar and dissimilar images. This helps the network to learn representations that are invariant to small changes in the input data.

2. Contrastive loss function: SimCLR uses a contrastive loss function to train the network to learn representations that are close together for similar images and far apart for dissimilar images. The contrastive loss function encourages the network to learn a meaningful representation of the images that captures the underlying structure of the images.



#### Data Augmentation

Choosing the right data augmentation techniques is crucial for the success of SimCLR. The choice of data augmentation techniques will depend on the characteristics of the data and the goals of the analysis. 



Some common data augmentation techniques used in SimCLR include random cropping, random flipping, random color distortion, and random Gaussian blur. 



As already discussed, these augmentations encode the invariances that we want the model to learn, such as invariance to translation, rotation, and color changes. These might not all apply to your data!

#### Contrastive Loss Function

SimCLR uses the **NT-Xent** loss (Normalized Temperature-scaled Cross-Entropy), an InfoNCE-style loss applied to two augmented views of the same image:

$$
L_{i,j} = -\log\left(\frac{\exp(z_i \cdot z_j / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(z_i \cdot z_k / \tau)}\right)
$$

where $z_i$, $z_j$ are the projected representations of a positive pair, the denominator runs over all $2N$ samples in the batch (so each anchor sees $2N{-}2$ negatives), and $\tau$ is a temperature parameter.

The original paper benchmarks NT-Xent against margin-based and logistic-style contrastive losses; NT-Xent wins, largely because the softmax over the full batch gives it many negatives "for free".

The temperature $\tau$ controls how sharply the loss focuses on the hardest negatives. Small $\tau$ amplifies the contribution of the closest negative samples. Typical values are $\tau \in [0.07, 0.5]$.

On top of the backbone, SimCLR adds a small **projection head** (a 2-layer MLP). The contrastive loss is computed on the *projected* embeddings, but it is the **pre-projection** backbone features that get used for downstream tasks.

The projection head over-specialises to the contrastive task, while the backbone retains more general-purpose information. Throwing the head away after training is one of SimCLR's most-cited findings.

## Tile2Vec

Tile2Vec is another technique for learning visual representations of imagery using unsupervised learning. The technique is based on the idea of learning a representation of the spatial context of the image tiles, rather than the content of the tiles themselves.


You can read more about the algorithm in the original 2018 paper: https://arxiv.org/abs/1805.02855


Tile2Vec works by training a convolutional neural network to predict the spatial context of the image tiles. For each training image, a nearby and a distant image are chosen to act as similar and dissimilar examples. A contrastive loss is then used to map them nearby and far away in the embedding space.



The result is a set of visual representations of the image tiles that capture the spatial context of the tiles. These representations can be used for a variety of tasks, such as image retrieval, image classification, and image segmentation.



## Tile2Vec

Tile2Vec is a powerful technique for learning visual representations of satellite imagery and has been shown to outperform other techniques for satellite image retrieval and classification.



The tile2vec algorithm uses a triplet loss, where the network is trained to minimize the distance between similar pairs of data points and maximize the distance between dissimilar pairs of data points:

$$
L = \sum_{i=1}^{N} \max(0, \alpha + d(f(x_i), f(x_i^+)) - d(f(x_i), f(x_i^-)))
$$

where $ d $ is a distance function, $ f $ is a function that maps the data points to the embedding space, $ x_i $ is a data point, $ x_i^+ $ is a similar data point, $ x_i^- $ is a dissimilar data point, and $ \alpha $ is a margin that separates the similar and dissimilar pairs.

Note the contrast with SimCLR's NT-Xent: triplet loss uses **one** negative per anchor, while NT-Xent uses **all other samples in the batch** as negatives. The richer negative set is part of why InfoNCE-style losses have largely displaced triplet loss for general representation learning.

Triplet-style methods remain compelling, however, when the geometry of the problem provides a natural positive-pair signal, as it does here with spatial proximity.

### Defining "similar" is where domain knowledge enters

The contrastive *framework* is general; the *positive-pair definition* is where you encode what you know about your data:

 - **SimCLR**: similar = same image under different augmentations
 - **Tile2Vec**: similar = spatially nearby tiles
 - **CLIP** (we'll see shortly): similar = matched image–caption pairs
 - **Time-series SSL**: similar = nearby time windows

## Applying Contrastive Learning to Remote Sensing

Let's take a look at an example of applying tile2vec to satellite imagery, and see how it can be used to learn visual representations of the image tiles.

One of the main focuses in my lab is better understanding the role of clouds in the climate system. Clouds come in many shapes and sizes, and understanding their properties is crucial for improving climate models and predicting future climate change.

You might be familiar with different types of clouds, such as cumulus clouds, stratus clouds, and cirrus clouds. Each type of cloud has its own unique properties, such as shape, size, and altitude.

<center><img src="_images/wmo_clouds.jpg" alt="clouds" width="800"/></center>

Only recently though we have started to explore the mesoscale morphology of clouds, which is the structure of clouds at scales of a few kilometers to a few hundred kilometers. This is important because the mesoscale morphology of clouds can have a significant impact on the Earth's energy balance and climate - and we don't really understand what controls it.

Bjorn Stevens' lab recently published a paper classifying cloud types based on their mesoscale morphology. He used a large dataset of satellite images of clouds and trained a convolutional neural network to classify the images into different cloud types (using a lot of labeled data).

The classes they used were Sugar, Flower, Fish, and Gravel. These are not the traditional cloud types you might be familiar with, but rather the mesoscale morphology of the clouds:

<center><img src="_images/sugar_gravel_flower_fish.png" alt="clouds" width="700"/></center>

In our Lab we have been using tile2vec on satellite imagery to learn visual representations of the mesoscale morphology of clouds *without* labels. The idea is to learn a representation of the spatial context of the cloud tiles, rather than the content of the tiles themselves.

We want to capture the important morphological features of the clouds, across a broad range of conditions and scales, so we use a false color representation of the images to highlight the different temperature (and hence altitude) of the clouds.

Here is an animation of the kind of data we're using:

<center><img src="_images/G16_DayNightCloudMicroCombo_240fr_20240528.gif" alt="clouds" width="700"/></center>

We're using a very simple ResNet architecture to learn the representations:

<center><img src="_images/tile2vec_architecture.png" alt='architecture' width=700/></center>

The key ingredient is how we choose the triplet - we pick a random tile from the training set, a nearby tile from the same image, and a dissimilar tile from a different image. This encodes the fact that we expect spatially nearby tiles to be similar, and distant tiles to different (in terms of the morphology of the clouds).

For example:

<center><img src="_images/example_triplet.png" alt='triplet' width=700/></center>

By training over 900,000 such triplets (and testing on a further 100,000), we can learn a representation of the mesoscale morphology of clouds that can be used for a variety of tasks.

The first thing we looked at was clustering the representations to see if we could identify different cloud types based on the morphology alone. We used k-means clustering to cluster the representations into distinct clusters, exploring the number of clusters from 9 to 30.

Already with N=9 we can see distinct clusters emerging:

<center><img src="_images/cluster_examples_9.png" alt='clusters' width=700/></center>

We can plot a t-SNE visualization of the representations to see how well the clusters are separated in the embedding space:

<center><img src="_images/example_tSNE.png" alt='tsne' width=700/></center>

We can also look at how the clusters are located geographically to see if they correspond to different regions of the world:

<center><img src="_images/cluster_map.png" alt='map' width=700/></center>

As you can see, these initial clusters are primarily based on the altitude of the cloud (based on its temperature) and the underlying surface  - we need more clusters to capture the full range of cloud types.

Another cool thing we can do is to find the nearest neighbors of a given tile in the embedding. This is like a reverse image search, where we search for similar images based on their visual content:

<center><img src="_images/similarity_search.png" alt='neighbors' width=700/></center>

And, even better - we can interpolate between any two tiles in the embedding space to see how the morphology of the clouds changes between them:

<center><img src="_images/interpolate_between_examples.png" alt='interpolation' width=800/></center>

Finally, we used the learnt model weights as the backbone of a classifier to predict the cloud type based on the morphology of the clouds with a few hundred labeled examples. 

Using our Tile2Vec pre-trained model:
 - Frozen weights: Accuracy → 0.4874
 - Unfrozen (fine-tuned) weights: Accuracy → 0.7395

Off-the-shelf ResNet (ImageNet-pretrained):
 - Frozen weights: Accuracy → 0.5462
 - Unfrozen (fine-tuned) weights: Accuracy → 0.7059

So this is really a comparison between **domain-matched self-supervised pretraining** (Tile2Vec on cloud imagery) and **out-of-domain supervised pretraining** (ImageNet). With fine-tuning, the domain-matched SSL backbone wins.

## Beyond SimCLR

SimCLR (2020) and Tile2Vec (2018) launched modern contrastive learning. Since then, three lines of work have reshaped the landscape:

 - **Can we get rid of negative pairs?** They force huge batch sizes and careful sampling.

 - **Why doesn't the network just collapse** to a constant embedding?

 - **Is contrastive the only way?** Or are there simpler self-supervised signals?

### Removing the negatives: BYOL & SimSiam

 - **BYOL** (Grill et al., 2020): match an "online" encoder's output to a slowly-updated target encoder. No negatives at all.
 - **SimSiam** (Chen & He, 2020): same encoder for both branches, but with a **stop-gradient** on one side.

The trick that prevents collapse is the **asymmetry + stop-gradient**, not the negatives.

### SimSiam in pictures

<center><img src="_images/simsiam_diagram.svg" alt="SimSiam architecture" width="720"/></center>

 - One image, two augmented views ($x_1$, $x_2$); the **same encoder $f$** processes both.
 - A **predictor $h$** sits on top of one branch only. This asymmetry is essential.
 - **Stop-gradient** on the other branch: $z_2$ acts as a frozen target.
 - Loss is negative cosine similarity; the full SimSiam loss averages two such terms with the branches swapped.

Removing either the predictor *or* the stop-gradient → immediate representation collapse. BYOL is essentially the same picture, but with a separately-parameterised target encoder updated via EMA instead of a stop-gradient.

### Self-distillation: DINO / DINOv2

 - **DINO** (Caron et al., 2021): student/teacher setup, teacher weights are an exponential moving average of the student. No labels, no negatives.
 - **DINOv2** (Oquab et al., 2023): scaled to 142M curated images.

Striking property: clean **object-aware attention maps** emerge without any labels. DINOv2 is now the de-facto strong vision SSL backbone, and likely what you would reach for first today.

### DINO in pictures

<center><img src="_images/dino_diagram.svg" alt="DINO architecture" width="720"/></center>

 - **Two networks, same architecture, separate weights.** The teacher's parameters $\xi$ are an exponential moving average of the student's $\theta$, with no teacher gradients and no labels.
 - Both networks output a probability distribution over a fixed-size codebook.
 - The student is trained to **match the teacher's distribution** via cross-entropy.
 - Collapse is prevented by **centering** (subtracting a running mean) and **sharpening** (low teacher temperature $\tau_t$). Centering kills the trivial "always output the same vector" solution; sharpening keeps the teacher targets confident.

DINOv2 keeps the same recipe but scales up to a 142M-image curated dataset with a ViT backbone, producing features strong enough to use as a frozen feature extractor across many vision tasks.

### A different paradigm: Masked Autoencoders (MAE)

He et al., 2021: mask out ${\sim}75\%$ of image patches and ask the network to reconstruct them.

 - **Not contrastive at all**. It is a reconstruction loss, much like the autoencoders from last lecture.
 - Simple, scales beautifully, and is the basis of most current **geoscience foundation models**.

### MAE in pictures

<center><img src="_images/mae_diagram.svg" alt="MAE architecture" width="740"/></center>

 - Image is split into patches; **~75% are masked at random**.
 - A **large ViT encoder** sees *only the visible patches*, making pretraining roughly $3{-}4\times$ cheaper than processing the full image.
 - A **lightweight decoder** then takes the encoded tokens together with learnable `[mask]` placeholders, and predicts pixel values for the masked patches.
 - Loss is **MSE on the masked patches only**.

The high masking ratio is what forces the encoder to learn *semantics* rather than just local pixel statistics. After pretraining, the decoder is discarded and the encoder is reused for downstream tasks, exactly analogous to throwing away SimCLR's projection head.

### Why does any of this work? Representation collapse

Naively, an SSL model could output the same vector for every input and trivially minimise any "similarity" loss. Each family has a different mechanism to prevent this:

| Family | Collapse-prevention mechanism |
|---|---|
| SimCLR, MoCo | Many negatives in each batch |
| BYOL, SimSiam | Stop-gradient + asymmetric branches |
| DINO | Centering + sharpening of teacher outputs |
| MAE | Pixel-level reconstruction target (cannot collapse) |

## From SSL to Foundation Models

The leap from Tile2Vec to today's "foundation models" is mostly:

 - Same self-supervised principles

 - ${\sim}1000\times$ more data

 - Transformer (ViT) backbones instead of small ResNets

The result: pre-trained models you can download and adapt to almost any task with a handful of labels.

### Vision–language contrastive: CLIP

Radford et al., 2021 (OpenAI):

 - Same NT-Xent loss, but the positive pairs are **(image, caption)** instead of (image, augmented-image).
 - Unlocks **zero-shot classification**: describe a class in text and ask which image matches.
 - The bridge between vision SSL and the LLM lecture coming up.

### Geoscience foundation models

Direct descendants of what we covered today, scaled up:

 - **SatCLIP** (Klemmer et al., 2023): CLIP for satellite imagery + geographic location.
 - **Prithvi** (NASA/IBM, 2023): MAE-based, pretrained on Harmonized Landsat–Sentinel data.
 - **Clay v1** (2024): open-source Earth observation foundation model, MAE backbone.
 - **SatMAE** (Cong et al., 2022): MAE adapted for multi-spectral satellite data.

All trained with the same SSL ideas we covered today.

### Tile2Vec's descendants: when to use what

Our cloud morphology work still uses Tile2Vec because:

 - False-color GOES imagery has no natural text captions → CLIP-style is out.
 - Spatial proximity is a meaningful, almost-free positive-pair signal.

For "standard" multi-spectral remote sensing tasks (land cover, flood detection, crop type), you would now reach for **Prithvi** or **Clay** off the shelf.

The principle is the same; the scale and machinery have grown.

### Practical takeaway

For most downstream geoscience vision tasks, the workflow is now:

1. **Download** a pretrained foundation model.
2. **Probe** it (train a linear classifier on frozen features) to see how well its representations match your task.
3. **Fine-tune** if you have enough labels and the gap is too large.

You will rarely train SSL from scratch, but you need to know the loss families above to read model cards, choose the right backbone, and interpret what it has and has not learned.

### Summary

We introduced **self-supervised learning** and looked at **contrastive learning** as one approach to it. Two canonical methods: **SimCLR** (augmentation-based positives, NT-Xent loss) and **Tile2Vec** (spatial-proximity positives, triplet loss).

We saw how Tile2Vec applied to satellite imagery learns meaningful representations of cloud mesoscale morphology, and outperforms ImageNet-pretrained features once fine-tuned.

We then looked **beyond SimCLR** (BYOL, DINO, MAE) and saw how the same SSL ideas, scaled up, underpin today's geoscience **foundation models** (CLIP, SatCLIP, Prithvi, Clay).

**The lesson**: pick your *positive-pair definition* to encode your domain knowledge; the rest of the contrastive framework is increasingly off-the-shelf.

### Next Week: Generative Models

Next week we will discuss generative models, which are a type of deep learning model that is used to generate new data samples from a given dataset. 



They differ from the deterministic models we have discussed so far, which are used to make predictions based on the input data. Generative models are used to generate new data samples that are similar to the training data by learning the underlying probabilistic distribution of the data.



We will discuss two popular generative models: Generative Adversarial Networks (GANs) and Variational Autoencoders (VAEs).